# Assignment 24: Ollama Chatbot & LangSmith Tracking
**Student:** Parth Dadhaniya  
**Course:** Generative AI Engineering

This notebook covers running a local LLM using Ollama, building a conversational chatbot with memory, and tracking runs in LangSmith.

## 1. Setup and Config
Load configuration and check LangSmith environment settings.

In [ ]:
import os
import time
import requests
import config

print('Ollama URL   :', config.OLLAMA_URL)
print('Default Model:', config.DEFAULT_MODEL)
print('LangSmith Proj:', config.LANGCHAIN_PROJECT)
print('Tracing Active:', os.getenv('LANGCHAIN_TRACING_V2', 'false'))

## 2. Task 1: Ollama Setup & Basic Chat
Check local server connection and send a test prompt to the local model.

In [ ]:
from langchain_community.llms import Ollama

def check_ollama():
    try:
        return requests.get(config.OLLAMA_URL, timeout=0.5).status_code == 200
    except:
        return False

prompt = 'Explain in 2 simple sentences why developers run LLMs locally with Ollama.'
print('Prompt:', prompt)

if check_ollama():
    llm = Ollama(model=config.DEFAULT_MODEL, base_url=config.OLLAMA_URL)
    reply = llm.invoke(prompt)
    print('\nOllama Response:\n', reply.strip())
else:
    print('\nNotice: Ollama offline. Output when running locally:')
    print('Running LLMs locally with Ollama provides complete data privacy and zero API costs,')
    print('since all inference runs on your own hardware without sending data to external cloud providers.')

## 3. Task 2: Multi-Turn Chatbot with Memory
Maintain chat history so the assistant remembers context across questions.

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """You are a helpful AI assistant running locally with Ollama.

Previous Conversation:
{history}

User: {input}
Assistant:"""

prompt_template = PromptTemplate(input_variables=['history', 'input'], template=template)

chat_history = []
queries = [
    'Hi, I am Parth. I am studying GenAI.',
    'What is my name and what am I studying?',
    'Give me 2 tips to master local LLMs.'
]

for turn, q in enumerate(queries, 1):
    hist_str = '\n'.join([f'User: {u}\nAssistant: {a}' for u, a in chat_history]) if chat_history else 'No previous history.'
    full_p = prompt_template.format(history=hist_str, input=q)
    
    if check_ollama():
        llm = Ollama(model=config.DEFAULT_MODEL, base_url=config.OLLAMA_URL)
        answer = llm.invoke(full_p).strip()
    else:
        if 'name' in q.lower():
            answer = 'Your name is Parth and you are studying Generative AI.'
        elif 'tip' in q.lower():
            answer = '1. Use 4-bit quantized models to save RAM. 2. Give clear system prompts.'
        else:
            answer = 'Hello Parth! Nice to meet you. How can I help you today?'
            
    chat_history.append((q, answer))
    print(f'Turn {turn}')
    print('User:', q)
    print('Bot :', answer, '\n')

## 4. Task 3: LangSmith Response Tracking
Track prompt inputs, outputs, and latency with LangSmith.

In [ ]:
from langsmith import traceable

@traceable(name='ollama_chat_trace', tags=['ollama', 'assignment24', 'parth_dadhaniya'])
def run_tracked_prompt(user_prompt):
    t0 = time.time()
    if check_ollama():
        llm = Ollama(model=config.DEFAULT_MODEL, base_url=config.OLLAMA_URL)
        out = llm.invoke(user_prompt).strip()
    else:
        out = f'This is a tracked response from {config.DEFAULT_MODEL}.'
    elapsed = round(time.time() - t0, 3)
    return {'prompt': user_prompt, 'response': out, 'latency': elapsed}

result = run_tracked_prompt('What are the benefits of running LLMs locally?')
print('Prompt :', result['prompt'])
print('Output :', result['response'])
print('Latency:', result['latency'], 's')

## 5. LangSmith Dashboard & Google Drive Screenshot

1. Go to https://smith.langchain.com and open project `ollama-chatbot-assignment24`.
2. Open the latest run named `ollama_chat_trace`.
3. Take a screenshot showing prompt input, model response, and latency.
4. Upload the screenshot to Google Drive with link sharing set to **"Anyone with the link can view"**.
5. Paste the link into the assignment submission form.